# NB_patient_360

Builds `<catalog>.gold.patient_360` (one row per patient) from silver patients, hospitals, insurance_providers and aggregated lab_results, per the STTM `Gold/sttm_silver_to_gold.xlsx`. Load type: FULL rebuild (CREATE OR REPLACE TABLE ... AS SELECT). All joins are LEFT JOINs from patients.

In [0]:
# Catalog placeholder from the STTM (<catalog>); can be overridden by ADF / Jobs
dbutils.widgets.text("catalog", "adb_caresync")
catalog = dbutils.widgets.get("catalog").strip() or "adb_caresync"

silver = f"{catalog}.silver"
target_table = f"{catalog}.gold.patient_360"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")
print(f"Source schema : {silver}")
print(f"Target table  : {target_table}")

In [0]:
# Notes from silver schema analysis:
#  * patients.date_of_birth is a TIMESTAMP in silver -> cast to DATE for the target
#  * silver.insurance_providers contains exact duplicate rows per insurance_provider_id
#    (appended loads) -> de-duplicate (latest update_timestamp) before joining, so the
#    one-row-per-patient grain is preserved. hospitals is de-duplicated the same way defensively.
sql_text = f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per patient with hospital, insurance and lab-test summary (source: STTM silver_to_gold)'
AS
WITH hospitals AS (
  SELECT hospital_id, hospital_name
  FROM {silver}.hospitals
  QUALIFY ROW_NUMBER() OVER (PARTITION BY hospital_id ORDER BY update_timestamp DESC) = 1
),
insurance_providers AS (
  SELECT insurance_provider_id, provider_name
  FROM {silver}.insurance_providers
  QUALIFY ROW_NUMBER() OVER (PARTITION BY insurance_provider_id ORDER BY update_timestamp DESC) = 1
),
lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                          AS total_tests,
    COUNT_IF(result_status = 'Abnormal')              AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')              AS critical_test_count,
    MAX(test_date)                                    AS last_test_date
  FROM {silver}.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                              AS patient_id,
  p.first_name                                                           AS first_name,
  p.last_name                                                            AS last_name,
  CAST(p.date_of_birth AS DATE)                                          AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender                                                               AS gender,
  p.city                                                                 AS city,
  p.state                                                                AS state,
  CAST(p.registered_hospital_id AS INT)                                  AS hospital_id,
  h.hospital_name                                                        AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                   AS insurance_provider_id,
  ip.provider_name                                                       AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                        AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                        AS critical_test_count,
  CAST(l.last_test_date AS DATE)                                         AS last_test_date
FROM {silver}.patients p
LEFT JOIN hospitals h            ON p.registered_hospital_id = h.hospital_id
LEFT JOIN insurance_providers ip ON p.insurance_provider_id  = ip.insurance_provider_id
LEFT JOIN lab_agg l              ON p.patient_id             = l.patient_id
"""
print(sql_text)
spark.sql(sql_text)
print(f"Created {target_table}")

In [0]:
src_cnt = spark.table(f"{silver}.patients").count()
validation = spark.sql(f"""
  SELECT
    COUNT(*)                                       AS total_rows,
    COUNT(DISTINCT patient_id)                     AS distinct_patients,
    {src_cnt}                                      AS source_patients,
    SUM(total_tests)                               AS total_tests,
    (SELECT COUNT(*) FROM {silver}.lab_results)    AS source_lab_results,
    COUNT_IF(hospital_name IS NULL)                AS missing_hospital_name,
    COUNT_IF(insurance_provider_name IS NULL)      AS missing_insurance_name
  FROM {target_table}
""")
display(validation)

r = validation.first()
assert r.total_rows == r.distinct_patients == r.source_patients, "Grain violated: expected one row per patient"
print("Validation passed: one row per patient, no rows dropped.")

In [0]:
# Return the gold record count to the caller (ADF / Jobs / dbutils.notebook.run)
updated_df = spark.table(target_table)
record_count = updated_df.count()
print(f"{target_table} record count: {record_count}")
dbutils.notebook.exit(str(record_count))